In [ ]:
import pandas as pd

mitochondria_prot = pd.read_csv("//wsl.localhost/Ubuntu/home/cezar/algo/project/subcell_location_Mitochondria.tsv", sep="\t")
cytosol_prot = pd.read_csv("//wsl.localhost/Ubuntu/home/cezar/algo/project/subcell_location_Aggresome_Cytosol_Cytoplasmic.tsv", sep="\t")

     Gene                                   Gene synonym          Ensembl  \
0  A4GALT                        A14GALT, Gb3S, P(k), P1  ENSG00000128274   
1   AARS2                    AARSL, bA444E17.1, KIAA1270  ENSG00000124608   
2    AASS                                 LKRSDH, LORSDH  ENSG00000008311   
3    AATK  AATYK, AATYK1, KIAA0641, LMR1, LMTK1, PPP1R77  ENSG00000181409   
4    ABAT                                  GABA-T, GABAT  ENSG00000183044   

                                  Gene description Uniprot Chromosome  \
0  Alpha 1,4-galactosyltransferase (P blood group)  Q9NPC4         22   
1          Alanyl-tRNA synthetase 2, mitochondrial  Q5JTZ9          6   
2               Aminoadipate-semialdehyde synthase  Q9UDR5          7   
3             Apoptosis associated tyrosine kinase  Q6ZMQ8         17   
4                 4-aminobutyrate aminotransferase  P80404         16   

              Position                                      Protein class  \
0    42692121-4272129

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import requests

url = "https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29"

response = requests.get(url)

with open("human_sprot.fasta", "w") as f:
    f.write(response.text)

print("Downloaded human_sprot.fasta")



Downloaded human_sprot.fasta


In [14]:
# Build a UniProt Dictionary
from Bio import SeqIO

seq_dict = {}

for record in SeqIO.parse("human_sprot.fasta", "fasta"):
    accession = record.id.split("|")[1]
    seq_dict[accession] = str(record.seq)[:40]

print(f"Loaded {len(seq_dict)} protein sequences")

Loaded 20431 protein sequences


In [16]:
mitochondria_prot["Uniprot_clean"] = (
    mitochondria_prot["Uniprot"]
    .astype(str)
    .str.split("-")
    .str[0]
)

cytosol_prot["Uniprot_clean"] = (
    cytosol_prot["Uniprot"]
    .astype(str)
    .str.split("-")
    .str[0]
)

mitochondria_prot["AA_sequence_40"] = (
    mitochondria_prot["Uniprot_clean"]
    .map(seq_dict)
)

cytosol_prot["AA_sequence_40"] = (
    cytosol_prot["Uniprot_clean"]
    .map(seq_dict)
)

In [17]:
print(
    mitochondria_prot[
        ["Gene", "Uniprot", "AA_sequence_40"]
    ].head()
)

print(
    f"Mitochondrial matches: "
    f"{mitochondria_prot['AA_sequence_40'].notna().sum()} / {len(mitochondria_prot)}"
)

print(
    f"Cytosolic matches: "
    f"{cytosol_prot['AA_sequence_40'].notna().sum()} / {len(cytosol_prot)}"
)

     Gene Uniprot                            AA_sequence_40
0  A4GALT  Q9NPC4  MSKPPDLLLRLLRGAPRQRVCTLFIIGFKFTFFVSIMIYW
1   AARS2  Q5JTZ9  MAASVAAAARRLRRAIRRSPAWRGLSHRPLSSEPPAAKAS
2    AASS  Q9UDR5  MLQVHRTGLGRLGVSLSKGLHHKAVLAVRREDVNAWERRA
3    AATK  Q6ZMQ8  MSSSFFNPSFAFSSHFDPDGAPLSELSWPSSLAVVAVSFS
4    ABAT  P80404  MASMLLAQRLACSFQHSYRLLVPGSRHISQAAAKVDVEFD
Mitochondrial matches: 1112 / 1132
Cytosolic matches: 5241 / 5341


In [18]:
full_seq_dict = {}

for record in SeqIO.parse("human_sprot.fasta", "fasta"):
    accession = record.id.split("|")[1]
    full_seq_dict[accession] = str(record.seq)

mitochondria_prot["AA_sequence"] = mitochondria_prot["Uniprot_clean"].map(full_seq_dict)
mitochondria_prot["AA_sequence_40"] = mitochondria_prot["AA_sequence"].str[:40]